# 01b - Agent Blueprint Initialization with Agent365 CLI

> **📍 Path A: Agent Identity Blueprint (Alternative)** - This notebook provides an alternative to `a365.ps1`.
> For AI Services demos (Azure Foundry, Search), skip to notebook 05.

**Learning objectives**
- Install and configure the Microsoft Agent 365 CLI
- Initialize agent configuration using the CLI instead of PowerShell scripts
- Understand the agent365 CLI workflow for agent development
- Validate CLI configuration and prepare for agent publishing

**Prerequisites**
- Python 3.11+ (3.13 recommended)
- .NET 8.0 SDK installed on your system
- Azure subscription with appropriate permissions
- Custom client app registration in Microsoft Entra ID
- Required roles: Global Administrator, Agent ID Administrator, or Agent ID Developer

**How to use this notebook**
- This notebook provides an alternative to the `a365.ps1` PowerShell script
- Uses the official agent365 CLI tool instead of direct Graph API calls
- Follows Microsoft's recommended workflow for agent development
- Run cells from top to bottom to initialize your agent environment

---

## Choose Your Setup Method

| Method | Best For | Config Output |
|--------|----------|---------------|
| **`a365.ps1` (Notebook 01)** | Learning, full control | `.env` (Section 2) |
| **`a365` CLI (This notebook)** | Teams publishing workflow | `a365.config.json` + `.env` |

**⚠️ Configuration Note:**
- Both methods work with `.env` for notebook validation
- The CLI also creates `a365.config.json` for Teams publishing
- Notebooks 01-04 read from `.env` (Section 2: Agent Identity Blueprint)
- Notebooks 05-08 are INDEPENDENT of agent identity setup

## What is the Agent365 CLI?

The **Agent365 CLI** (`a365`) is Microsoft's official command-line tool for developing and managing Microsoft Agent 365 applications.

**Key differences from PowerShell script approach:**

| Aspect | `a365.ps1` (PowerShell) | `a365` CLI (This notebook) |
|--------|-------------------------|---------------------------|
| **Approach** | Low-level Graph API calls | High-level abstraction |
| **Control** | Full control over each step | Streamlined workflow |
| **Focus** | Blueprint creation | Publish/deploy workflow |
| **Output** | `.env` file | `a365.config.json` + `.env` |
| **Use Case** | Learning, customization | Teams publishing |

**Agent365 CLI capabilities:**
- `a365 config init`: Initialize and validate agent configuration
- `a365 publish`: Publish agent to Microsoft Teams
- `a365 deploy`: Deploy agent to Azure (if using Azure Web App hosting)

**Important note:**
The agent365 CLI assumes you already have an app registration (Agent Identity Blueprint) created. It validates and configures the existing app rather than creating it from scratch. For creating the initial blueprint, you can:
1. Use the `a365.ps1` PowerShell script (Step 1-4)
2. Manually create it in Azure Portal
3. Use Azure CLI or Terraform

**Documentation reference:**
- [Agent 365 CLI Overview](https://learn.microsoft.com/en-us/microsoft-agent-365/developer/agent-365-cli)
- [Agent development workflow](https://learn.microsoft.com/en-us/microsoft-agent-365/developer/quickstart)

## Step 1: Check .NET SDK Installation

The agent365 CLI requires .NET 8.0 (recommended) to be installed. Let's verify the installation.

In [ ]:
%%bash
# Check if dotnet is installed
if command -v dotnet &> /dev/null; then
    echo "✅ .NET SDK is installed"
    echo ""
    dotnet --version
    echo ""
    dotnet --list-sdks
else
    echo "❌ .NET SDK is not installed"
    echo ""
    echo "Please install .NET 8.0 SDK from:"
    echo "https://dotnet.microsoft.com/download/dotnet/8.0"
    echo ""
    echo "Installation commands:"
    echo "  macOS:   brew install dotnet@8"
    echo "  Linux:   See https://learn.microsoft.com/en-us/dotnet/core/install/linux"
    echo "  Windows: Download installer from Microsoft"
    exit 1
fi

## Step 2: Install Agent365 CLI

Install the Microsoft Agent 365 CLI as a global .NET tool.

**Note:** The tool is in prerelease, so we need to use the `--prerelease` flag.

In [ ]:
%%bash
echo "📦 Installing Microsoft Agent 365 CLI..."
echo ""

# Install the CLI tool globally
dotnet tool install --global Microsoft.Agents.A365.DevTools.Cli --prerelease

# Check installation
if [ $? -eq 0 ]; then
    echo ""
    echo "✅ Agent365 CLI installed successfully!"
else
    echo ""
    echo "⚠️  Installation may have failed or tool is already installed"
    echo "    If already installed, you can update with:"
    echo "    dotnet tool update --global Microsoft.Agents.A365.DevTools.Cli --prerelease"
fi

## Step 3: Verify Agent365 CLI Installation

Let's verify the CLI is installed correctly and check available commands.

In [ ]:
%%bash
echo "🔍 Verifying Agent365 CLI installation..."
echo ""

# Display help to verify installation
a365 -h

## Step 4: Configure Environment Variables

Before running `a365 config init`, we need to ensure we have the necessary environment variables set up.

**Required for agent365 CLI:**
- Application (Client) ID from your app registration
- Tenant ID
- Client Secret or Certificate credentials

**Two scenarios:**
1. **If you ran a365.ps1**: Load existing .env configuration
2. **If starting fresh**: You'll need to create an app registration first

Let's check for existing configuration:

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Check for existing .env file
env_path = Path(".env")

if env_path.exists():
    print("✅ Found existing .env file")
    load_dotenv(env_path)
    
    tenant_id = os.getenv("AZURE_TENANT_ID")
    client_id = os.getenv("AZURE_CLIENT_ID")
    credential_type = os.getenv("AZURE_CLIENT_CREDENTIAL_TYPE")
    
    print("\n📋 Loaded Configuration:")
    print(f"   Tenant ID: {tenant_id}")
    print(f"   Client ID: {client_id}")
    print(f"   Credential Type: {credential_type}")
    
    if tenant_id and client_id:
        print("\n✅ Configuration looks good! Ready to initialize agent365 CLI.")
    else:
        print("\n⚠️  Configuration incomplete. Please run a365.ps1 first or create .env manually.")
else:
    print("❌ No .env file found")
    print("\n⚠️  You need to create an Agent Identity Blueprint first.")
    print("\nOptions:")
    print("  1. Run the a365.ps1 PowerShell script:")
    print("     pwsh a365.ps1 -Step menu")
    print("\n  2. Manually create app registration in Azure Portal:")
    print("     - Go to Azure Portal → Entra ID → App registrations")
    print("     - Create new registration for Agent Identity Blueprint")
    print("     - Configure API permissions and admin consent")
    print("     - Create client secret or certificate")
    print("\n  3. Use Azure CLI to create app registration")
    print("\nThen create a .env file with:")
    print("  AZURE_TENANT_ID=<your-tenant-id>")
    print("  AZURE_CLIENT_ID=<your-client-id>")
    print("  AZURE_CLIENT_SECRET=<your-client-secret>  # if using secret auth")

## Step 5: Initialize Agent365 Configuration

Run `a365 config init` to initialize and validate the agent configuration.

**What this command does:**
- Validates your custom client app registration exists
- Checks app has required permissions
- Verifies admin consent is granted
- Creates local configuration for the agent project

**Prerequisites:**
- App registration must already exist in Entra ID
- App must have required API permissions
- Admin consent must be granted

**Note:** This is an interactive command that will prompt for authentication.

In [ ]:
%%bash
# Note: This command is interactive and requires user authentication
# It will open a browser window for Azure AD authentication

echo "🔐 Initializing agent365 configuration..."
echo ""
echo "⚠️  This will open a browser window for authentication."
echo ""

# Run config init
# Note: In a notebook environment, interactive commands may not work as expected
# You may need to run this in a terminal instead

# Uncomment the line below to run (or run in terminal):
# a365 config init

echo "ℹ️  To run this step, execute the following in your terminal:"
echo "   a365 config init"
echo ""
echo "This will:"
echo "  1. Prompt for Azure AD authentication"
echo "  2. Validate your app registration"
echo "  3. Check required permissions"
echo "  4. Verify admin consent"
echo "  5. Create local agent configuration"

## Step 6: Agent365 CLI Workflow Overview

After initialization, here's the typical agent365 CLI workflow:

### 1. Develop Your Agent Code
Create your agent application using the Microsoft Agent SDK or framework of choice.

### 2. Publish Agent
```bash
a365 publish
```
- Packages your agent
- Uploads to Microsoft Teams
- Makes it available in Teams Developer Portal

### 3. Configure in Teams Developer Portal
- Open Teams Developer Portal
- Configure agent blueprint settings
- Set capabilities, permissions, etc.

### 4. Deploy to Azure (Optional)
```bash
a365 deploy
```
- Only needed if hosting on Azure Web App
- Deploys agent code to Azure

### 5. Create Agent Instance in Teams
- Open Microsoft Teams
- Navigate to Apps
- Search for your agent
- Click "Add" to create instance

**Important Notes:**
- The `create-instance` command was removed from the CLI (it bypassed required registration steps)
- Agent instances must be created through Microsoft Teams
- Users need "Microsoft Agent 365 Frontier" license to create instances

## Comparison: PowerShell Script vs Agent365 CLI

### PowerShell Script (a365.ps1)
**Pros:**
- ✅ Complete control over blueprint creation
- ✅ Creates all components from scratch
- ✅ Can be automated in CI/CD pipelines
- ✅ Works without Teams Developer Portal
- ✅ Good for infrastructure-as-code

**Cons:**
- ❌ Requires PowerShell and Microsoft.Graph module
- ❌ Lower-level API interactions
- ❌ More complex error handling
- ❌ Need to manage all Graph API details

**Use when:**
- Creating agent infrastructure programmatically
- Need full control over all settings
- Automating agent deployment in CI/CD
- Working with multiple environments

### Agent365 CLI
**Pros:**
- ✅ Official Microsoft tool
- ✅ Streamlined developer workflow
- ✅ Integrated with Teams Developer Portal
- ✅ Automatic validation and checks
- ✅ Simple command-line interface

**Cons:**
- ❌ Requires .NET 8.0
- ❌ Currently in prerelease
- ❌ Assumes app registration already exists
- ❌ Less control over low-level settings
- ❌ Interactive authentication may not work in all environments

**Use when:**
- Following standard agent development workflow
- Publishing agents to Microsoft Teams
- Working with Teams Developer Portal
- Deploying to Azure Web App

### Recommended Approach
**Best practice: Combine both approaches**

1. **Initial Setup (Infrastructure):**
   - Use `a365.ps1` to create Agent Identity Blueprint
   - Set up service principals, credentials, OAuth scopes
   - Store configuration in .env file

2. **Development & Publishing:**
   - Use `a365 config init` to validate configuration
   - Use `a365 publish` to publish agent to Teams
   - Use Teams Developer Portal for agent configuration
   - Use `a365 deploy` for Azure deployments

This gives you:
- Full control over infrastructure (PowerShell)
- Streamlined development workflow (CLI)
- Best of both worlds

## Environment Setup Script

Here's a helper script to set up your environment for agent365 CLI usage:

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

# Load environment from .env if it exists
env_path = Path(".env")
if not env_path.exists():
    print("⚠️  No .env file found. Please run a365.ps1 first or create one manually.")
else:
    load_dotenv(env_path)
    
    # Create a configuration summary for agent365 CLI
    config = {
        "agent365_cli": {
            "installed": True,  # We assume this after running install cell
            "version": "prerelease",
            "install_command": "dotnet tool install --global Microsoft.Agents.A365.DevTools.Cli --prerelease",
            "update_command": "dotnet tool update --global Microsoft.Agents.A365.DevTools.Cli --prerelease",
            "uninstall_command": "dotnet tool uninstall --global Microsoft.Agents.A365.DevTools.Cli"
        },
        "environment": {
            "tenant_id": os.getenv("AZURE_TENANT_ID"),
            "client_id": os.getenv("AZURE_CLIENT_ID"),
            "credential_type": os.getenv("AZURE_CLIENT_CREDENTIAL_TYPE")
        },
        "next_steps": [
            "Run 'a365 config init' in terminal to initialize configuration",
            "Develop your agent code using Microsoft Agent SDK",
            "Run 'a365 publish' to publish agent to Teams",
            "Configure agent in Teams Developer Portal",
            "(Optional) Run 'a365 deploy' to deploy to Azure",
            "Create agent instance in Microsoft Teams"
        ]
    }
    
    print("📊 Agent365 CLI Configuration Summary:\n")
    print(json.dumps(config, indent=2))
    
    # Save configuration
    with open("agent365-cli-config.json", "w") as f:
        json.dump(config, f, indent=2)
    
    print("\n💾 Configuration saved to: agent365-cli-config.json")
    
    # Display next steps
    print("\n🎯 Next Steps:")
    for i, step in enumerate(config["next_steps"], 1):
        print(f"   {i}. {step}")

## CLI Command Reference

### Installation & Updates
```bash
# Install
dotnet tool install --global Microsoft.Agents.A365.DevTools.Cli --prerelease

# Update
dotnet tool update --global Microsoft.Agents.A365.DevTools.Cli --prerelease

# Uninstall
dotnet tool uninstall --global Microsoft.Agents.A365.DevTools.Cli

# Verify installation
a365 -h
```

### Configuration
```bash
# Initialize agent configuration
a365 config init
```

### Publishing
```bash
# Publish agent to Microsoft Teams
a365 publish
```

### Deployment
```bash
# Deploy to Azure Web App (if using Azure hosting)
a365 deploy
```

### Getting Help
```bash
# Show help
a365 -h

# Show help for specific command
a365 publish -h
a365 deploy -h
a365 config -h
```

## Troubleshooting

### Common Issues

**1. .NET SDK not found**
- **Error**: `dotnet: command not found`
- **Solution**: Install .NET 8.0 SDK from https://dotnet.microsoft.com/download/dotnet/8.0

**2. Tool installation fails**
- **Error**: Failed to install tool
- **Solution**: Ensure you have internet connectivity and NuGet is accessible
- Check .NET tool installation path:
  - Linux/macOS: `$HOME/.dotnet/tools`
  - Windows: `%USERPROFILE%\.dotnet\tools`

**3. a365 command not found after installation**
- **Problem**: PATH not updated
- **Solution**: Add .NET tools directory to PATH or restart terminal
```bash
# macOS/Linux
export PATH="$PATH:$HOME/.dotnet/tools"

# Windows (PowerShell)
$env:PATH += ";$env:USERPROFILE\.dotnet\tools"
```

**4. Config init fails with authentication error**
- **Problem**: App registration missing or permissions not granted
- **Solution**: 
  1. Verify app registration exists in Azure Portal
  2. Check required API permissions are configured
  3. Ensure admin consent is granted
  4. Run `a365.ps1 -Step menu` to complete setup

**5. Interactive commands don't work in notebook**
- **Problem**: Jupyter notebooks don't support interactive CLI tools well
- **Solution**: Run `a365` commands in a terminal instead
  - Open terminal
  - Navigate to notebook directory
  - Run commands directly

**6. License requirement error**
- **Error**: User needs "Microsoft Agent 365 Frontier" license
- **Solution**: Contact your Azure admin to assign the required license

## Summary

### What You've Learned

✅ **Agent365 CLI Basics**
- Installed and configured the Microsoft Agent 365 CLI
- Understood the CLI workflow for agent development
- Learned the differences between PowerShell and CLI approaches

✅ **Environment Setup**
- Verified .NET SDK installation
- Installed agent365 CLI as global .NET tool
- Configured environment variables for CLI usage

✅ **Workflow Understanding**
- Learned the agent development lifecycle
- Understood when to use PowerShell vs CLI
- Know how to combine both approaches effectively

### Key Takeaways

1. **Agent365 CLI** is the official tool for agent development workflows
2. **PowerShell script** provides low-level control for infrastructure setup
3. **Combine both** for the best experience:
   - PowerShell for initial blueprint creation
   - CLI for development, publishing, and deployment
4. **Interactive commands** should be run in terminal, not notebook
5. **License requirement**: Users need "Microsoft Agent 365 Frontier" license

### Next Steps

1. **Run in Terminal**: Execute `a365 config init` in your terminal to complete initialization
2. **Develop Agent**: Create your agent code using Microsoft Agent SDK
3. **Publish**: Use `a365 publish` to publish to Microsoft Teams
4. **Configure**: Use Teams Developer Portal to configure agent settings
5. **Deploy**: (Optional) Use `a365 deploy` if hosting on Azure

### Additional Resources

- [Agent 365 CLI Documentation](https://learn.microsoft.com/en-us/microsoft-agent-365/developer/agent-365-cli)
- [Agent Development Quickstart](https://learn.microsoft.com/en-us/microsoft-agent-365/developer/quickstart)
- [Microsoft Agent SDK](https://learn.microsoft.com/en-us/microsoft-agent-365/developer/sdk)
- [Teams Developer Portal](https://dev.teams.microsoft.com/)
- [Agent 365 CLI Reference](https://learn.microsoft.com/en-us/microsoft-agent-365/developer/agent-365-cli/reference/cli/)